In [1]:
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize

In [2]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\prana\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\prana\AppData\Roaming\nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\prana\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\prana\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
def detect_ambiguity(sentence):
    print(f"Original Sentence: {sentence}\n")

    # 1. Tokenize the sentence
    tokens = word_tokenize(sentence)

    # 2. Identify Nouns and Verbs (POS Tagging)
    # NNP/NN/NNS are Nouns; VB/VBD/VBG/VBN/VBP/VBZ are Verbs
    pos_tags = nltk.pos_tag(tokens)
    
    ambiguous_words = []

    print(f"{'Word':<15} | {'POS':<10} | {'Sense Count':<15} | {'Sample Meaning'}")
    print("-" * 75)

    for word, tag in pos_tags:
        # Filter for Nouns (N) and Verbs (V)
        if tag.startswith('NN') or tag.startswith('VB'):
            # 3. Use WordNet to extract synsets (meanings)
            synsets = wordnet.synsets(word)
            
            # 4. Print words having more than one meaning
            if len(synsets) > 1:
                sample_meaning = synsets[0].definition()
                print(f"{word:<15} | {tag:<10} | {len(synsets):<15} | {sample_meaning}")
                
                ambiguous_words.append({
                    'word': word,
                    'count': len(synsets),
                    'definition': sample_meaning
                })

    return ambiguous_words

In [9]:
test_sentence = "The crane is flying over the construction site."
results = detect_ambiguity(test_sentence)

if not results:
    print("\nNo significant lexical ambiguity detected.")

Original Sentence: The crane is flying over the construction site.

Word            | POS        | Sense Count     | Sample Meaning
---------------------------------------------------------------------------
crane           | NN         | 6               | United States writer (1871-1900)
is              | VBZ        | 13              | have the quality of being; (copula, used with an adjective or a predicate noun)
flying          | VBG        | 17              | an instance of traveling by air
construction    | NN         | 7               | the act of constructing something
site            | NN         | 4               | the piece of land on which something is located (or is to be located)


In [4]:
import nltk
from nltk.corpus import wordnet
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize

# Setup
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

def analyze_contextual_ambiguity(sentences):
    # 1. Identification
    tokens = word_tokenize(sentences[0])
    pos_tags = nltk.pos_tag(tokens)
    
    ambiguous_word = None
    max_senses = 0
    
    for word, tag in pos_tags:
        if tag.startswith(('NN', 'VB')):
            senses = len(wordnet.synsets(word))
            if senses > max_senses:
                max_senses = senses
                ambiguous_word = word.lower() # Ensure lowercase for WordNet

    print(f"--- [TASK 1] Identification of Ambiguous Word ---")
    print(f"Detected Word: '{ambiguous_word}' ({max_senses} meanings found)\n")

    # 2, 3 & 4. Extraction and Explanation
    print(f"--- [TASK 2, 3 & 4] Analysis ---")
    for i, sent in enumerate(sentences, 1):
        tokens_sent = word_tokenize(sent)
        # We pass pos='n' to force it to look at Noun meanings (Geography/Finance)
        sense = lesk(tokens_sent, ambiguous_word, pos='n')
        
        print(f"Context {i}: \"{sent}\"")
        if sense:
            definition = sense.definition()
            print(f"Extracted Meaning: {definition}")
            
            definition_lower = definition.lower()
            geo_keys = ['river', 'water', 'land', 'slope', 'side', 'shore', 'stream', 'adjoining']
            fin_keys = ['money', 'financial', 'institution', 'funds', 'gambling', 'deposits', 'cash']

            if any(key in definition_lower for key in geo_keys):
                print(f"Result: The word '{ambiguous_word}' is used in a GEOGRAPHY context.")
            elif any(key in definition_lower for key in fin_keys):
                print(f"Result: The word '{ambiguous_word}' is used in a FINANCE/BUSINESS context.")
            else:
                print("Result: Category identified outside Finance/Geography.")
        print("-" * 60)

# Dataset with the "clue" word 'river'
data = ["The fisherman waited by the river bank."]

analyze_contextual_ambiguity(data)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\prana\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\prana\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\prana\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


--- [TASK 1] Identification of Ambiguous Word ---
Detected Word: 'bank' (18 meanings found)

--- [TASK 2, 3 & 4] Analysis ---
Context 1: "The fisherman waited by the river bank."
Extracted Meaning: the funds held by a gambling house or the dealer in some gambling games
Result: The word 'bank' is used in a FINANCE/BUSINESS context.
------------------------------------------------------------


In [5]:
import nltk
from nltk import CFG

# 1. Define a Context-Free Grammar (CFG)
# The ambiguity here is created by the 'PP' (Prepositional Phrase) 
# which can attach to either the Verb Phrase (VP) or the Noun Phrase (NP).
grammar = CFG.fromstring("""
  S -> NP VP
  PP -> P NP
  NP -> Det N | Det N PP | 'i'
  VP -> V NP | V NP PP
  Det -> 'the' | 'a'
  N -> 'man' | 'telescope' | 'dog'
  V -> 'saw' | 'followed'
  P -> 'with'
""")

def detect_syntactic_ambiguity(sentence):
    print(f"Analyzing Sentence: {sentence}\n")
    
    # 2. Use NLTK ChartParser
    tokens = sentence.lower().split()
    parser = nltk.ChartParser(grammar)
    
    # 3. Generate all possible parse trees
    trees = list(parser.parse(tokens))
    
    print(f"--- [TASK 3] Generated {len(trees)} Parse Trees ---\n")
    
    # 4. Display and Explain Trees
    for i, tree in enumerate(trees, 1):
        print(f"--- TREE {i} ---")
        tree.pretty_print()
        
        # 4. Explanation of interpretations
        if i == 1:
            print("Interpretation 1: The Prepositional Phrase (with the telescope) modifies the VERB.")
            print("Meaning: The action of 'seeing' was done using the telescope.\n")
        else:
            print("Interpretation 2: The Prepositional Phrase (with the telescope) modifies the NOUN.")
            print("Meaning: The 'man' being seen was carrying or had a telescope.\n")

# --- Execution ---
sent = "I saw the man with the telescope"
detect_syntactic_ambiguity(sent)

Analyzing Sentence: I saw the man with the telescope

--- [TASK 3] Generated 2 Parse Trees ---

--- TREE 1 ---
     S                                    
  ___|___________                          
 |               VP                       
 |    ___________|________                 
 |   |       |            PP              
 |   |       |        ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 NP  V  Det      N   P   Det         N    
 |   |   |       |   |    |          |     
 i  saw the     man with the     telescope

Interpretation 1: The Prepositional Phrase (with the telescope) modifies the VERB.
Meaning: The action of 'seeing' was done using the telescope.

--- TREE 2 ---
     S                                
  ___|_______                          
 |           VP                       
 |    _______|___                      
 |   |           NP                   
 |   |    _______|____                 
 |   |   |  